# Scheduling, batching, and scaling

## Learning goals
- Find the bottleneck stage in a heterogeneous pipeline
- Scale a stage's replicas and measure the makespan drop
- See diminishing returns once the bottleneck moves

> **Mac track.** Every executed cell below is a pure-Python simulation that runs on any Mac with no GPU. Cells labelled **Linux GPU lab** show the *real* vLLM-Omni commands — they are shown as text and are **not** run here, because the official `vllm serve --omni` runtime is CUDA-only.

## The slowest stage sets the throughput

In a pipeline, the stage with the lowest sustained throughput (replicas / service_time) is the bottleneck. For Qwen3-Omni that is the **Talker**: it decodes ~3.6x more tokens than the text head. We feed the voice pipeline's service times into a tandem-queue simulator and locate it.

In [ ]:
from omni_tutorial import build_voice_pipeline, PipelineSimulator
specs = build_voice_pipeline().stage_specs()
arrivals = [i * 0.5 for i in range(40)]
result = PipelineSimulator(specs).run(arrivals)
print('bottleneck:', result['bottleneck'])
for s in result['stage_stats']:
    flag = '  <-- bottleneck' if s['bottleneck'] else ''
    print(f"{s['stage']:8} mean_wait={s['mean_wait']:5.1f} capacity={s['capacity']:.2f}{flag}")

## Scale the bottleneck

We sweep the Talker's replica count and watch the makespan (time to clear all 40 requests) and the Talker's own max wait fall — then flatten once a different stage becomes limiting.

In [ ]:
from omni_tutorial import sweep_replicas
rows = sweep_replicas(specs, 'talker', [1, 2, 3, 4, 5, 6], arrivals)
for r in rows:
    print(f"talker x{r['replicas']}: makespan={r['makespan']:6.1f} "
          f"talker_max_wait={r['stage_max_wait']:6.1f} bottleneck={r['bottleneck']}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from omni_tutorial import sweep_replicas
rows = sweep_replicas(specs, 'talker', list(range(1, 9)), arrivals)
reps = [r['replicas'] for r in rows]
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(reps, [r['makespan'] for r in rows], 'o-', label='makespan')
ax.plot(reps, [r['stage_max_wait'] for r in rows], 's--', label='talker max wait')
ax.set_xlabel('talker replicas'); ax.set_ylabel('time (arb. units)')
ax.set_title('Scaling the bottleneck: diminishing returns'); ax.legend()
fig.tight_layout(); plt.show()

## Exercise

After enough Talker replicas the bottleneck moves. Which stage does it move to, and why doesn't adding *more* Talkers help past that point?

In [ ]:
# Solution: once talker capacity exceeds the others, the Thinker (service_time=1.0,
# 1 replica) becomes limiting. Extra talkers then sit idle waiting for thinker output.
rows = sweep_replicas(specs, 'talker', [1, 4, 8], arrivals)
for r in rows:
    print(f"talker x{r['replicas']}: bottleneck is now {r['bottleneck']}")

## Source lab

Find per-stage scheduling and the `--stage-overrides` handling in `vllm_omni/`. Confirm each stage engine owns its own scheduler and replica set.